# Photo-z training catalog: EUCLID_DR1 bbox → crossmatch → AB → FITS

Build an AB-homogenized, FITS-exported photometric catalog for **EUCLID_DR1** sources in a bounding box, joined with IR/optical partner photometry and spectroscopic redshift + quality flags where available. Suitable for photometric redshift fitting of galaxies and AGN.

**Environment:** [uv](https://docs.astral.sh/uv/) project venv (`uv sync --extra dev`). Kernel **Python (data-lake)**.

**Pipeline:**

1. Verify bbox coverage and column names
2. Save sky region as an area (`dl-region --bbox --save-as`)
3. Attach crossmatch + gather plans (`dl-area`)
4. Positional crossmatch (`dl-crossmatch --from-area`)
5. Wide native product (`dl-gather --from-area` → `products/<name>/`)
6. Write EUCLID `phot_ab_v1` recipe + homogenize to AB (`dl-homogenize --from-product`)
7. Export FITS (`dl-extract-catalog`)
8. (Optional) Extract DESI 1D spectra via `dl-extract-spectra-subset` using `DESI_DR1_TARGETID` from the gathered product (no re-gather)

Companion docs: [`docs/discovery/workflow.md`](../docs/discovery/workflow.md), [`docs/homogenization.md`](../docs/homogenization.md).

> Set `$DATA_LAKE_CONFIG` to your deployment before running. Heavy steps (crossmatch, gather, homogenize) are **commented out** by default — uncomment when ready to execute on the full box.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path

# ── Deployment ───────────────────────────────────────────────────────────────
# export DATA_LAKE_CONFIG=/path/to/your/lake_config.toml
if not os.environ.get("DATA_LAKE_CONFIG"):
    raise EnvironmentError(
        "Set DATA_LAKE_CONFIG to your lake_config.toml before running this notebook."
    )

from data_lake.config import LakeConfig

cfg = LakeConfig.discover()
LAKE_ROOT = Path(cfg.lake.root)
SCRATCH = Path("/scratch/euclid_photoz")  # change to a writable path
SCRATCH.mkdir(parents=True, exist_ok=True)

# ── Field parameters (edit for your science case) ────────────────────────────
RA_MIN, RA_MAX = 179.0, 181.0   # degrees
DEC_MIN, DEC_MAX = -1.5, 1.5    # degrees
AREA_ID = "euclid_photoz_box"
PRODUCT_NATIVE = "euclid_photoz_native_v1"
PRODUCT_AB = "euclid_photoz_ab_v1"
N_WORKERS = 8

print("Lake root:", LAKE_ROOT)
print("Config:", os.environ["DATA_LAKE_CONFIG"])
print(f"BBox: RA [{RA_MIN}, {RA_MAX}]  Dec [{DEC_MIN}, {DEC_MAX}]")

In [ ]:
def run_dl(*args: str, check: bool = True) -> subprocess.CompletedProcess:
    """Run a dl-* CLI command and print it. On failure, surface stderr/stdout."""
    cmd = ["dl-" + args[0]] + list(args[1:])
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if check and proc.returncode != 0:
        details = (proc.stderr or proc.stdout or "").strip()
        raise RuntimeError(
            f"{' '.join(cmd)} failed (exit {proc.returncode}):\n{details}"
        )
    return proc


def show(cmd: subprocess.CompletedProcess, max_lines: int = 40) -> None:
    if cmd.stdout:
        lines = cmd.stdout.splitlines()
        print("\n".join(lines[:max_lines]))
        if len(lines) > max_lines:
            print(f"… ({len(lines) - max_lines} more lines)")
    if cmd.stderr:
        print(cmd.stderr, end="")

## Step 0 — Verify inputs (read-only)

Confirm which surveys cover the bbox and inspect column names before defining the gather plan. Bundled `phot_ab_v1` recipes exist for ALLWISE, VIKING_DR4, GAIA_DR3_source, etc. **EUCLID_DR1 has no bundled recipe** — we add a lake override in Step 5.

In [ ]:
# Tile overlap and row counts in the bbox
show(run_dl(
    "region", str(LAKE_ROOT),
    "--bbox", str(RA_MIN), str(RA_MAX), str(DEC_MIN), str(DEC_MAX),
    "--count",
))

In [ ]:
# Column manifests for base + partners (truncated output)
SURVEYS = [
    "EUCLID_DR1",
    "ALLWISE",
    "VIKING_DR4",
    "GAIA_DR3_source",
    "DESI_DR1",
    "SDSS_DR17",
]
for survey in SURVEYS:
    p = LAKE_ROOT / "catalogs" / survey
    if not p.is_dir():
        print(f"SKIP {survey} (not ingested)")
        continue
    out = run_dl("describe-survey", survey, "--modality", "catalog")
    print(out.stdout.splitlines()[0:6])
    print()

In [ ]:
# Which ingested catalogs lack phot_ab_v1 recipes?
show(run_dl("validate-homogenization", "--ab-coverage"), max_lines=30)

## Step 1 — Save the bbox region as an area

In [ ]:
area_path = LAKE_ROOT / "areas" / f"{AREA_ID}.json"
if area_path.exists():
    print(f"Area already exists: {area_path}")
else:
    show(run_dl(
        "region", str(LAKE_ROOT),
        "--bbox", str(RA_MIN), str(RA_MAX), str(DEC_MIN), str(DEC_MAX),
        "--save-as", AREA_ID,
    ))
    print(f"Created {area_path}")

## Step 2 — Attach crossmatch + gather plans

Base = **EUCLID_DR1**. Mix sky and column partners on the same `set-crossmatch`:

- **Sky:** `SURVEY:RADIUS_ARCSEC` — e.g. `ALLWISE:2.0`, `VIKING_DR4:1.0`, `GAIA_DR3_source:1.0`
- **Column equality:** `SURVEY:col:COL_A:COL_B` — e.g. `DESI_DR1:col:TARGETID:TARGETID` (COL_A on base, COL_B on partner; names from `dl-describe-survey`; confirm via `dl-describe-lake` detail `col:COL_A:COL_B`)

Below: IR/optical by sky; DESI/SDSS by shared ID (edit COL_A/COL_B if your EUCLID catalog uses different names). Spec-z columns are prefixed at gather time (`DESI_DR1_Z`, `DESI_DR1_ZWARN`, …). Include **`TARGETID`** in the DESI gather columns so Step 8 can map to DESI spectra (`DESI_DR1_TARGETID` in the product).

In [ ]:
GATHER_COLUMNS = {
    "EUCLID_DR1": [
        "right_ascension", "declination", "object_id",
        "flux_vis_2fwhm_aper", "fluxerr_vis_2fwhm_aper",
        "flux_y_2fwhm_aper", "fluxerr_y_2fwhm_aper",
        "flux_j_2fwhm_aper", "fluxerr_j_2fwhm_aper",
        "flux_h_2fwhm_aper", "fluxerr_h_2fwhm_aper",
        "flux_nir_stack_2fwhm_aper", "fluxerr_nir_stack_2fwhm_aper",
        "flux_g_ext_decam_2fwhm_aper", "fluxerr_g_ext_decam_2fwhm_aper",
        "flux_r_ext_decam_2fwhm_aper", "fluxerr_r_ext_decam_2fwhm_aper",
        "flux_i_ext_decam_2fwhm_aper", "fluxerr_i_ext_decam_2fwhm_aper",
        "flux_z_ext_decam_2fwhm_aper", "fluxerr_z_ext_decam_2fwhm_aper",
        "det_quality_flag", "flag_vis", "flag_y", "flag_j", "flag_h",
    ],
    "ALLWISE": [
        "w1mpro", "w1sigmpro", "w2mpro", "w2sigmpro",
        "w3mpro", "w3sigmpro", "w4mpro", "w4sigmpro",
    ],
    "VIKING_DR4": [
        "ZAPERMAG3", "ZAPERMAG3ERR",
        "YAPERMAG3", "YAPERMAG3ERR",
        "JAPERMAG3", "JAPERMAG3ERR",
        "HAPERMAG3", "HAPERMAG3ERR",
        "KSAPERMAG3", "KSAPERMAG3ERR",
    ],
    "GAIA_DR3_source": [
        "phot_g_mean_mag", "phot_bp_mean_mag", "phot_rp_mean_mag",
    ],
    "DESI_DR1": ["TARGETID", "Z", "ZWARN", "SPECTYPE"],
    "SDSS_DR17": ["Z", "ZWARNING", "CLASS"],
}

columns_json = json.dumps(GATHER_COLUMNS)
print(columns_json[:500], "…")

In [ ]:
show(run_dl(
    "area", str(LAKE_ROOT), "set-crossmatch", AREA_ID,
    "--base", "EUCLID_DR1",
    "--partner", "ALLWISE:2.0",
    "--partner", "VIKING_DR4:1.0",
    "--partner", "GAIA_DR3_source:1.0",
    # Column equality: SURVEY:col:COL_A:COL_B
    "--partner", "DESI_DR1:col:EUCLID_ID_COL:TARGETID",
    "--partner", "SDSS_DR17:col:EUCLID_ID_COL:SPECOBJID",
))

show(run_dl(
    "area", str(LAKE_ROOT), "set-gather", AREA_ID,
    "--base", "EUCLID_DR1",
    "--columns", columns_json,
    "--materialize-as", PRODUCT_NATIVE,
))

# Inspect the area file
print((LAKE_ROOT / "areas" / f"{AREA_ID}.json").read_text()[:2000], "…")

## Step 3 — Crossmatch (region-bounded, resumable)

Writes `crossmatch/EUCLID_DR1_x_<partner>__r<R>/` trees for tiles in the bbox only. **Uncomment to run** (can take hours on large boxes).

In [ ]:
RUN_CROSSMATCH = False  # set True to execute

if RUN_CROSSMATCH:
    show(run_dl(
        "crossmatch", str(LAKE_ROOT),
        "--from-area", AREA_ID,
        "--n-workers", str(N_WORKERS),
        "--progress",
    ), max_lines=60)
else:
    print("Skipped (set RUN_CROSSMATCH = True to run)")

## Step 4 — Gather the wide native product

Output: `products/euclid_photoz_native_v1/` with `_source_id` (EUCLID join key), native EUCLID columns, and prefixed partner columns (`ALLWISE_w1mpro`, `DESI_DR1_TARGETID`, `DESI_DR1_Z`, `DESI_DR1_ZWARN`, …). Default keep-all: every EUCLID source is retained; partner columns are null when unmatched.

> If you gathered an earlier product without `DESI_DR1_TARGETID`, re-run gather with `--overwrite` after updating `GATHER_COLUMNS` and the area JSON.

In [ ]:
RUN_GATHER = False  # set True after crossmatch completes

if RUN_GATHER:
    show(run_dl("gather", str(LAKE_ROOT), "--from-area", AREA_ID), max_lines=40)
    show(run_dl("describe-lake", "--kind", "product"), max_lines=20)
else:
    print("Skipped (set RUN_GATHER = True to run)")

## Step 5a — EUCLID_DR1 `phot_ab_v1` recipe (lake override)

EUCLID flux columns are in **µJy**. The `flux_to_ab` transform expects flux in Jy-equivalent form with `zp=23.9` for µJy inputs (`m_AB = -2.5 log10(f_µJy) + 23.9`). Verify against the Euclid DR1 calibration documentation for your release.

Partner surveys (ALLWISE, VIKING, GAIA) use bundled recipes in `data_lake/homogenize/surveys/`.

In [ ]:
EUCLID_RECIPE = {
    "survey": "EUCLID_DR1",
    "version": 1,
    "recipe_status": "ready",
    "notes": "Euclid DR1 2-fwhm aperture fluxes in microJy. zp=23.9 for flux_to_ab on uJy.",
    "catalog": {
        "phot_ab_v1": {
            "output_system": "AB_mag",
            "rules": [
                {
                    "source_column": "flux_vis_2fwhm_aper",
                    "target_column": "phot_ab_vis",
                    "uncertainty_column": "fluxerr_vis_2fwhm_aper",
                    "target_uncertainty_column": "phot_ab_vis_err",
                    "native_system": "uJy",
                    "transform": {"type": "flux_to_ab", "zp": 23.9},
                },
                {
                    "source_column": "flux_y_2fwhm_aper",
                    "target_column": "phot_ab_y",
                    "uncertainty_column": "fluxerr_y_2fwhm_aper",
                    "target_uncertainty_column": "phot_ab_y_err",
                    "native_system": "uJy",
                    "transform": {"type": "flux_to_ab", "zp": 23.9},
                },
                {
                    "source_column": "flux_j_2fwhm_aper",
                    "target_column": "phot_ab_j",
                    "uncertainty_column": "fluxerr_j_2fwhm_aper",
                    "target_uncertainty_column": "phot_ab_j_err",
                    "native_system": "uJy",
                    "transform": {"type": "flux_to_ab", "zp": 23.9},
                },
                {
                    "source_column": "flux_h_2fwhm_aper",
                    "target_column": "phot_ab_h",
                    "uncertainty_column": "fluxerr_h_2fwhm_aper",
                    "target_uncertainty_column": "phot_ab_h_err",
                    "native_system": "uJy",
                    "transform": {"type": "flux_to_ab", "zp": 23.9},
                },
                {
                    "source_column": "flux_nir_stack_2fwhm_aper",
                    "target_column": "phot_ab_nir_stack",
                    "uncertainty_column": "fluxerr_nir_stack_2fwhm_aper",
                    "target_uncertainty_column": "phot_ab_nir_stack_err",
                    "native_system": "uJy",
                    "transform": {"type": "flux_to_ab", "zp": 23.9},
                },
                {
                    "source_column": "flux_g_ext_decam_2fwhm_aper",
                    "target_column": "phot_ab_g_decam",
                    "uncertainty_column": "fluxerr_g_ext_decam_2fwhm_aper",
                    "target_uncertainty_column": "phot_ab_g_decam_err",
                    "native_system": "uJy",
                    "transform": {"type": "flux_to_ab", "zp": 23.9},
                },
                {
                    "source_column": "flux_r_ext_decam_2fwhm_aper",
                    "target_column": "phot_ab_r_decam",
                    "uncertainty_column": "fluxerr_r_ext_decam_2fwhm_aper",
                    "target_uncertainty_column": "phot_ab_r_decam_err",
                    "native_system": "uJy",
                    "transform": {"type": "flux_to_ab", "zp": 23.9},
                },
                {
                    "source_column": "flux_i_ext_decam_2fwhm_aper",
                    "target_column": "phot_ab_i_decam",
                    "uncertainty_column": "fluxerr_i_ext_decam_2fwhm_aper",
                    "target_uncertainty_column": "phot_ab_i_decam_err",
                    "native_system": "uJy",
                    "transform": {"type": "flux_to_ab", "zp": 23.9},
                },
                {
                    "source_column": "flux_z_ext_decam_2fwhm_aper",
                    "target_column": "phot_ab_z_decam",
                    "uncertainty_column": "fluxerr_z_ext_decam_2fwhm_aper",
                    "target_uncertainty_column": "phot_ab_z_decam_err",
                    "native_system": "uJy",
                    "transform": {"type": "flux_to_ab", "zp": 23.9},
                },
            ],
        }
    },
}

recipe_dir = LAKE_ROOT / "shared" / "registry" / "homogenize"
recipe_dir.mkdir(parents=True, exist_ok=True)
recipe_path = recipe_dir / "EUCLID_DR1.json"
recipe_path.write_text(json.dumps(EUCLID_RECIPE, indent=2) + "\n")
print(f"Wrote {recipe_path}")

## Step 5b — Homogenize photometry to AB

`--from-product` processes all tiles in the gathered product. Spec-z columns pass through unchanged.

In [ ]:
RUN_HOMOGENIZE = False  # set True after gather completes

if RUN_HOMOGENIZE:
    # Dry-run rule resolution
    show(run_dl(
        "homogenize",
        "--from-product", PRODUCT_NATIVE,
        "--transform", "phot_ab_v1",
        "--check-only",
    ))

    show(run_dl(
        "homogenize",
        "--from-product", PRODUCT_NATIVE,
        "--transform", "phot_ab_v1",
        "--materialize-as", PRODUCT_AB,
    ), max_lines=40)

    show(run_dl(
        "validate-homogenization",
        "--product", PRODUCT_AB,
        "--transform", "phot_ab_v1",
    ))
else:
    print("Skipped (set RUN_HOMOGENIZE = True to run)")

## Step 6 — Export to FITS

Writes a single merged FITS table plus a homogenize provenance sidecar. Use `-c` flags instead of `--all-columns` to project a subset for ML.

In [ ]:
RUN_EXPORT = False  # set True after homogenize completes
OUTPUT_FITS = SCRATCH / f"{PRODUCT_AB}.fits"

if RUN_EXPORT:
    show(run_dl(
        "extract-catalog",
        "--survey", PRODUCT_AB,
        "--all-columns",
        "-o", str(OUTPUT_FITS),
        "--format", "fits",
        "--progress",
    ), max_lines=30)
    print(f"Exported: {OUTPUT_FITS}")
    prov = OUTPUT_FITS.with_suffix(OUTPUT_FITS.suffix + ".homogenize_provenance.json")
    if prov.exists():
        print(f"Provenance: {prov}")
else:
    print(f"Skipped (set RUN_EXPORT = True; output would be {OUTPUT_FITS})")

## Optional — DESI spectra bundle for SED checks

Extract 1-D DESI spectra for sources that matched in the gathered product. **Does not re-run `dl-gather`.**

EUCLID `_source_id` in the product is **not** the DESI lake join key. This step:

1. Reads distinct `DESI_DR1_TARGETID` values from `PRODUCT_NATIVE` (requires `TARGETID` in gather columns).
2. Calls `dl-extract-spectra-subset` on `DESI_DR1` with `--with-catalog` for fast `_spectrum_index` lookup.

DESI catalog/spectra must be ingested with `--link-id-col TARGETID` so `TARGETID` matches lake `_source_id`. Apply `--apply-survey-calibration` for flux in physical units.

In [ ]:
RUN_EXTRACT_SPECTRA = False
SPECTRA_DIR = SCRATCH / "euclid_photoz_spectra"
EXTRACT_SPECTRA_SURVEY = "DESI_DR1"
DESI_PARTNER_ID_COL = "DESI_DR1_TARGETID"  # prefixed TARGETID from gather
DESI_ID_LIST = SCRATCH / "desi_targetids.parquet"
SPECTRA_ZARR = SPECTRA_DIR / "desi_spectra.zarr"
# Optional quality filter when building the ID list (set False to take all non-null TARGETIDs)
DESI_SPECTRA_REQUIRE_ZWARN0 = True

if RUN_EXTRACT_SPECTRA:
    from data_lake.io.catalog import CatalogAccessor
    from data_lake.schema_registry import resolve_catalog_root

    product_root = resolve_catalog_root(LAKE_ROOT, PRODUCT_NATIVE)
    if not (product_root / "catalog_info.json").is_file():
        raise FileNotFoundError(
            f"Gathered product not found: {product_root}\n"
            f"Run Step 4 (RUN_GATHER = True) first."
        )

    where = f"WHERE {DESI_PARTNER_ID_COL} IS NOT NULL"
    if DESI_SPECTRA_REQUIRE_ZWARN0:
        where += " AND DESI_DR1_ZWARN = 0"

    with CatalogAccessor(LAKE_ROOT, PRODUCT_NATIVE) as acc:
        if DESI_PARTNER_ID_COL not in acc.columns:
            raise ValueError(
                f"Column {DESI_PARTNER_ID_COL!r} not in {PRODUCT_NATIVE}. "
                "Add TARGETID to DESI_DR1 gather columns and re-gather with --overwrite."
            )
        id_df = acc.query(
            f"SELECT DISTINCT {DESI_PARTNER_ID_COL} AS TARGETID FROM catalog {where}",
            fmt="polars",
        )

    SPECTRA_DIR.mkdir(parents=True, exist_ok=True)
    id_df.write_parquet(DESI_ID_LIST)
    print(f"Wrote {len(id_df):,} DESI TARGETID(s) → {DESI_ID_LIST}")

    show(run_dl(
        "extract-spectra-subset",
        "--survey", EXTRACT_SPECTRA_SURVEY,
        "--target-list", str(DESI_ID_LIST),
        "--target-id-col", "TARGETID",
        "--format", "zarr",
        "--output", str(SPECTRA_ZARR),
        "--lake-root", str(LAKE_ROOT),
        "--apply-survey-calibration",
        "--progress",
        "--overwrite",
    ), max_lines=30)
    print(f"Spectra bundle: {SPECTRA_ZARR}")
else:
    print(
        f"Skipped (set RUN_EXTRACT_SPECTRA = True; "
        f"needs {PRODUCT_NATIVE} with {DESI_PARTNER_ID_COL}; "
        f"output {SPECTRA_ZARR})"
    )

## Science notes

- **Spec-z merge (downstream):** prefer DESI (`DESI_DR1_Z` when `DESI_DR1_ZWARN == 0`), fall back to SDSS (`SDSS_DR17_Z` when `SDSS_DR17_ZWARNING == 0`). The lake keeps separate provenance columns.
- **DESI spectra:** use `DESI_DR1_TARGETID` from the gathered product with `dl-extract-spectra-subset` — not EUCLID `_source_id`. Re-gather if the product predates adding `TARGETID` to the gather plan.
- **AGN:** `nearest` match at 1″ is appropriate for point sources; use `--multiplicity all` in a second gather if you need to audit blended matches.
- **Extended sources:** use `ALLWISE_sep_arcsec` to flag ambiguous WISE matches at the 2″ radius.
- **Photo-z training:** use rows with valid spec-z as training/validation; rows with null spec-z are photo-z targets. Filter on `det_quality_flag` and band flags before fitting.
- **EUCLID calibration:** confirm µJy zero-point and aperture choice (2-fwhm used here) against the Euclid DR1 release notes before publication.

In [ ]:
# Example: load exported FITS and inspect spec-z coverage (after RUN_EXPORT)
from astropy.table import Table

if OUTPUT_FITS.exists():
    tab = Table.read(OUTPUT_FITS)
    print(f"Rows: {len(tab):,}  columns: {len(tab.colnames)}")
    z_cols = [c for c in tab.colnames if "Z" in c.upper() or "ZWARN" in c.upper()]
    print("Redshift-related columns:", z_cols)
else:
    print("No FITS yet — run export step first.")